In [ ]:
!pip install gradio reportlab autopep8 black -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.9/88.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 16.1 MB/s eta 0:00:00


In [ ]:
import ast
import gradio as gr
import autopep8
import black
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Preformatted
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib import colors
from reportlab.lib.units import inch
from reportlab.lib.pagesizes import A4
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.pdfbase import pdfmetrics
from datetime import datetime
import os

In [ ]:
class DocGenieAnalyzer:

    @staticmethod
    def extract_function_signature(code):
        tree = ast.parse(code)
        for node in tree.body:
            if isinstance(node, ast.FunctionDef):
                func_name = node.name

                params = []
                for arg in node.args.args:
                    param_type = None
                    if arg.annotation:
                        param_type = ast.unparse(arg.annotation)
                    params.append({
                        "name": arg.arg,
                        "type": param_type,
                        "default": None
                    })

                return_type = None
                if node.returns:
                    return_type = ast.unparse(node.returns)

                return {
                    "name": func_name,
                    "params": params,
                    "return_type": return_type,
                    "node": node
                }
        return None

    @staticmethod
    def analyze_function_logic(func_node):
        analysis = {
            "has_loops": False,
            "has_conditions": False,
            "operations": [],
            "function_calls": [],
            "returns": False
        }

        for node in ast.walk(func_node):
            if isinstance(node, (ast.For, ast.While)):
                analysis["has_loops"] = True

            if isinstance(node, ast.If):
                analysis["has_conditions"] = True

            if isinstance(node, ast.BinOp):
                if isinstance(node.op, ast.Add):
                    analysis["operations"].append("addition")
                elif isinstance(node.op, ast.Sub):
                    analysis["operations"].append("subtraction")
                elif isinstance(node.op, ast.Mult):
                    analysis["operations"].append("multiplication")
                elif isinstance(node.op, ast.Div):
                    analysis["operations"].append("division")

            if isinstance(node, ast.Call):
                if isinstance(node.func, ast.Name):
                    analysis["function_calls"].append(node.func.id)

            if isinstance(node, ast.Return):
                analysis["returns"] = True

        return analysis

    @staticmethod
    def generate_google_docstring(signature, analysis):
        summary = f"{signature['name'].replace('_',' ').capitalize()} function."

        details = "Executes function logic"
        if analysis["has_loops"]:
            details += " with loop processing"
        if analysis["has_conditions"]:
            details += " and conditional handling"
        details += "."

        doc = f'"""{summary}\n\n{details}\n\nArgs:\n'
        for param in signature["params"]:
            ptype = param["type"] if param["type"] else "Any"
            doc += f"    {param['name']} ({ptype}): Description of {param['name']}.\n"

        if signature["return_type"]:
            doc += f"\nReturns:\n    {signature['return_type']}: Description of return value.\n"

        doc += '"""'
        return doc

    @staticmethod
    def generate_numpy_docstring(signature, analysis):
        summary = f"{signature['name'].replace('_',' ').capitalize()} function."

        details = "Executes function logic."
        doc = f'"""{summary}\n\n{details}\n\nParameters\n----------\n'

        for param in signature["params"]:
            ptype = param["type"] if param["type"] else "Any"
            doc += f"{param['name']} : {ptype}\n    Description of {param['name']}.\n"

        if signature["return_type"]:
            doc += f"\nReturns\n-------\n{signature['return_type']}\n    Description of return value.\n"

        doc += '"""'
        return doc


In [ ]:
def generate_doc(code, style):
    try:
        signature = DocGenieAnalyzer.extract_function_signature(code)
        if not signature:
            return "No function detected."

        analysis = DocGenieAnalyzer.analyze_function_logic(signature["node"])

        if style == "Google":
            docstring = DocGenieAnalyzer.generate_google_docstring(signature, analysis)
        else:
            docstring = DocGenieAnalyzer.generate_numpy_docstring(signature, analysis)

        formatted_code = f"{docstring}\n{code}"
        return formatted_code

    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
def export_pdf(content):
    file_path = "docgenie_output.pdf"
    doc = SimpleDocTemplate(file_path, pagesize=A4)
    styles = getSampleStyleSheet()
    elements = []

    elements.append(Paragraph("<b>Doc-Genie Documentation</b>", styles["Title"]))
    elements.append(Spacer(1, 0.3 * inch))
    elements.append(Preformatted(content, styles["Code"]))

    doc.build(elements)
    return file_path

In [ ]:
def export_txt(content):
    file_path = "docgenie_output.txt"
    with open(file_path, "w") as f:
        f.write(content)
    return file_path

In [ ]:
with gr.Blocks(title="Doc-Genie") as demo:

    gr.Markdown("# 🧠 Doc-Genie - Python Docstring Generator")

    code_input = gr.Code(language="python", label="Enter Python Function Code")
    style = gr.Radio(["Google", "NumPy"], value="Google", label="Docstring Style")
    generate_btn = gr.Button("Generate Docstring")

    output = gr.Code(language="python", label="Output Code")

    pdf_btn = gr.Button("Download PDF")
    txt_btn = gr.Button("Download TXT")

    pdf_file = gr.File()
    txt_file = gr.File()

    generate_btn.click(generate_doc, inputs=[code_input, style], outputs=output)
    pdf_btn.click(export_pdf, inputs=output, outputs=pdf_file)
    txt_btn.click(export_txt, inputs=output, outputs=txt_file)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e89723c30e21b0c49c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
